# Estimación de peso de pollos

## Colegio de Posgraduados

### COA661 Inteligencia Artificial

Profesor: Dr. Juan Manuel González Camacho

Entrega: José Alfredo Martínez

En este notebook se entrenan 4 modelos para predecir el peso de los pollos: Support Vector Regressor, Random Forest Regressor, K Nearest Neighbors y Multilayer Perceptron.

Se utilizan 5 características del Dataset

In [1]:
# Librerías
import numpy as np
import pandas as pd

import optuna

from IPython.display import clear_output

from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import sklearn.model_selection

import joblib

## Código

In [2]:
# Cargar datos
datos1 = pd.read_excel('./misDatos.xlsx', sheet_name = 'X_train')
datos2 = pd.read_excel('./misDatos.xlsx', sheet_name = 'X_test')

fil, col = datos1.shape

X_train = datos1.iloc[:, [0, 1, 4, 5, 6]].to_numpy()
Y_train = datos1.iloc[:, col - 1].to_numpy()

X_test = datos2.iloc[:, [0, 1, 4, 5, 6]].to_numpy()
Y_test = datos2.iloc[:, col - 1].to_numpy()

In [3]:
# Cargar datos de estandarización
scx = joblib.load('T_scaler5.pkl')

In [4]:
# Estandarizar los datos
X_train_std = scx.transform(X_train)
X_test_std = scx.transform(X_test)

print(X_train_std.shape)

(2439, 5)


## Support Vector Regressor

In [5]:
# Optimización con Optuna
def objective(trial):
    clear_output(wait=True)
    
    kernel = trial.suggest_categorical('kernel', ['rbf'])
    C = trial.suggest_float('C', 0.1, 100, log = True)
    gamma = trial.suggest_float('gamma', 0.001, 10, log = True)
    epsilon = trial.suggest_float('epsilon', 0.0001, 1)

    modelo = SVR(kernel = kernel, C = C, gamma = gamma, epsilon = epsilon)
    # Crear el modelo y realizar la Cross Validation
    modelo = SVR(kernel = kernel, C = C, gamma = gamma, epsilon = epsilon)
    r2_cv = sklearn.model_selection.cross_val_score(modelo, X_train_std, Y_train, n_jobs = -1, cv = 5, scoring = 'r2').mean()
    
    return r2_cv

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 200)

# Imprimir los mejores hiperparámetros y R2
print('Cross Validation')
print('Mejores hiperparámetros: ', study.best_params)
print('R2: ', round(study.best_value, 2))

[I 2026-05-18 23:28:21,811] Trial 199 finished with value: 0.7005915444939675 and parameters: {'kernel': 'rbf', 'C': 9.573284349805936, 'gamma': 0.03922135441291138, 'epsilon': 0.06043494819184986}. Best is trial 161 with value: 0.8510397072077863.


Cross Validation
Mejores hiperparámetros:  {'kernel': 'rbf', 'C': 5.166720024721651, 'gamma': 9.959644183979375, 'epsilon': 0.000998403323118353}
R2:  0.85


In [6]:
# Guardar datos de modelo
svr = SVR(**study.best_params)
joblib.dump(svr, './modelo_svr5.pkl')

['./modelo_svr5.pkl']

## Random Forest Regressor

In [7]:
# Optimización con Optuna
def objective(trial):
    clear_output(wait=True)
    
    n_estimators = trial.suggest_int('n_estimators', 100, 500)
    max_depth = trial.suggest_int('max_depth', 2, 35)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)

    modelo = RandomForestRegressor(n_estimators = n_estimators, max_depth = max_depth, min_samples_split = min_samples_split)
    r2_cv = sklearn.model_selection.cross_val_score(modelo, X_train, Y_train, n_jobs = -1, cv = 5, scoring = 'r2').mean()
    
    return r2_cv

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 200)

# Imprimir los mejores hiperparámetros y R2
print('Cross Validation')
print('Mejores hiperparámetros: ', study.best_params)
print('R2: ', round(study.best_value, 2))

[I 2026-05-18 23:35:27,415] Trial 199 finished with value: 0.8762030969973811 and parameters: {'n_estimators': 455, 'max_depth': 25, 'min_samples_split': 2}. Best is trial 65 with value: 0.8775588583853635.


Cross Validation
Mejores hiperparámetros:  {'n_estimators': 436, 'max_depth': 18, 'min_samples_split': 2}
R2:  0.88


In [8]:
# Guardar datos de modelo
rf = RandomForestRegressor(**study.best_params)
joblib.dump(rf, './modelo_rf5.pkl')

['./modelo_rf5.pkl']

## K Neighbors Regressor

In [9]:
def objective(trial):
    clear_output(wait=True)
    
    n_neighbors = trial.suggest_int('n_neighbors', 1, 50)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    metric = trial.suggest_categorical('metric', ['euclidean', 'manhattan'])

    modelo = KNeighborsRegressor(n_neighbors = n_neighbors, weights = weights, metric = metric)
    r2_cv = sklearn.model_selection.cross_val_score(modelo, X_train_std, Y_train, n_jobs = -1, cv = 5, scoring = 'r2').mean()
    
    return r2_cv

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 100)

# Imprimir los mejores hiperparámetros y R2
print('Cross Validation')
print('Mejores hiperparámetros: ', study.best_params)
print('R2: ', round(study.best_value, 2))

[I 2026-05-18 23:35:29,481] Trial 99 finished with value: 0.8736223314347458 and parameters: {'n_neighbors': 8, 'weights': 'distance', 'metric': 'euclidean'}. Best is trial 22 with value: 0.8752189585246917.


Cross Validation
Mejores hiperparámetros:  {'n_neighbors': 6, 'weights': 'distance', 'metric': 'euclidean'}
R2:  0.88


In [10]:
# Guardar datos de modelo
knn = KNeighborsRegressor(**study.best_params)
joblib.dump(knn, './modelo_knn5.pkl')

['./modelo_knn5.pkl']

## Multilayer Perceptron Regressor

In [11]:
def objective(trial):
    clear_output(wait=True)

    hidden_layer_sizes = trial.suggest_categorical('hidden_layer_sizes', [(25,), (50,), (25, 25), (50, 25)])
    activation = trial.suggest_categorical('activation', ['relu', 'tanh'])
    alpha = trial.suggest_float('alpha', 0.00001, 0.1, log = True)
    learning_rate_init = trial.suggest_float('learning_rate_init', 0.0001, 0.1, log = True)

    modelo = MLPRegressor(hidden_layer_sizes = hidden_layer_sizes, activation = activation, alpha = alpha, learning_rate_init = learning_rate_init, max_iter = 2000, random_state = 42)
    r2_cv = sklearn.model_selection.cross_val_score(modelo, X_train_std, Y_train, n_jobs = -1, cv = 5, scoring = 'r2').mean()

    return r2_cv

study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials = 200)

# Imprimir los mejores hiperparámetros y R2
print('Cross Validation')
print('Mejores hiperparámetros: ', study.best_params)
print('R2: ', round(study.best_value, 2))

[I 2026-05-18 23:36:37,808] Trial 199 finished with value: 0.6939239430587233 and parameters: {'hidden_layer_sizes': (25, 25), 'activation': 'tanh', 'alpha': 2.4891166950015955e-05, 'learning_rate_init': 0.004418204214258005}. Best is trial 181 with value: 0.7395282058764973.


Cross Validation
Mejores hiperparámetros:  {'hidden_layer_sizes': (25, 25), 'activation': 'relu', 'alpha': 1.6410687950194025e-05, 'learning_rate_init': 0.005137990119657661}
R2:  0.74


In [12]:
# Guardar datos de modelo
mlp = MLPRegressor(**study.best_params)
joblib.dump(mlp, './modelo_mlp5.pkl')

['./modelo_mlp5.pkl']